# 📊 Sales Data Analysis: Exploratory Data Analysis & Business Insights
**Author:** Aryansh Upadhyay (BBA – Business Intelligence & Analytics, LNCT University)
**Project:** [sales-data-analysis](https://github.com/aryanshupadhyaywork/sales-data-analysis)

---
### 🎯 Objectives:
1. Ingest and profile retail transactional records.
2. Clean and engineer financial and temporal features.
3. Uncover category, regional, and customer segment drivers of profitability.
4. Evaluate discount sensitivity and provide data-backed strategic recommendations.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
plt.style.use("seaborn-v0_8-whitegrid") if "seaborn-v0_8-whitegrid" in plt.style.available else plt.style.use("ggplot")
print("Libraries loaded successfully.")

## 1. Data Ingestion & Overview

In [ ]:
data_path = os.path.join("..", "data", "raw", "sales_data_raw.csv")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2. Data Cleaning & Feature Engineering

In [ ]:
# Datetime parsing
df["OrderDate"] = pd.to_datetime(df["OrderDate"])
df["ShipDate"] = pd.to_datetime(df["ShipDate"])

# Temporal features
df["Year"] = df["OrderDate"].dt.year
df["Month"] = df["OrderDate"].dt.month
df["YearMonth"] = df["OrderDate"].dt.to_period("M").astype(str)
df["Quarter"] = "Q" + df["OrderDate"].dt.quarter.astype(str)
df["FulfillmentDays"] = (df["ShipDate"] - df["OrderDate"]).dt.days

# Financial metrics
df["ProfitMarginPct"] = np.where(df["Sales"] > 0, (df["Profit"] / df["Sales"]) * 100, 0.0)
df.describe()

## 3. Revenue & Profitability Analysis by Category

In [ ]:
category_summary = df.groupby("Category").agg(
    TotalSales=("Sales", "sum"),
    TotalProfit=("Profit", "sum"),
    TotalQuantity=("Quantity", "sum")
).reset_index()

category_summary["MarginPct"] = (category_summary["TotalProfit"] / category_summary["TotalSales"]) * 100
category_summary.sort_values(by="TotalSales", ascending=False)

## 4. Regional Performance Breakdown

In [ ]:
regional_summary = df.groupby("Region").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Transactions=("OrderID", "count")
).reset_index()
regional_summary["MarginPct"] = (regional_summary["Profit"] / regional_summary["Sales"]) * 100
regional_summary.sort_values(by="Profit", ascending=False)

## 5. Discount Impact Analysis

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=df, x="Discount", y="ProfitMarginPct", hue="Category", alpha=0.7)
plt.axhline(0, color="red", linestyle="--", linewidth=1.5, label="Breakeven (0% Margin)")
plt.title("Discount Depth vs. Profit Margin %", fontsize=13, fontweight="bold")
plt.xlabel("Discount Ratio (0.0 = 0%, 0.5 = 50%)")
plt.ylabel("Profit Margin (%)")
plt.legend()
plt.show()